In [102]:
from mootdx.quotes import Quotes
import pandas as pd
import plotly.express as px
import re

qf10='经营分析'
client = Quotes.factory(market='std')
txt = client.F10('300207', qf10)[116:]

In [103]:
re.findall(r'\d+\.\d+|\d+%',re.findall(r'前5大客户.*',txt)[0])

['228.76', '47.80']

In [104]:
re.findall(r'\d+\.\d+|\d+%',re.findall(r'前5大供应商.*',txt)[0])


['138.22', '40.55']

In [105]:
fi = txt[txt.find('【2.主营构成分析】'):]
ff = fi[:fi.find('【3.前5名客户营业收入表】')]
datetimes=re.findall(r'截止日期:\s{0}(\S{10})', ff)


In [106]:
dd = ff.replace('─','').splitlines(keepends=False)

In [107]:

# Data = pd.DataFrame(columns=("股票名称", "一周涨跌幅%","一月涨跌幅%","三月涨跌幅%","半年涨跌幅%","一年涨跌幅%"))
Data = pd.DataFrame(columns=())
i = 0
while i < len(dd):
    lis = re.split(r"\s+", dd[i])[-6:]
    if len(lis)!=6:
        i = i+1
        # pass
    else:
        df = pd.DataFrame(lis).T
        # df.columns=["股票名称", "一周涨跌幅%","一月涨跌幅%","三月涨跌幅%","半年涨跌幅%","一年涨跌幅%"]
        Data = pd.concat([Data, df],axis=0)
        i=i+1
Data.reset_index(drop=True,inplace=True)
Data = Data.replace('---',0)
ddf  = Data.apply(pd.to_numeric, errors='ignore')

In [108]:
ddfindex = ddf[ddf[0]=='项目名'].index

In [109]:
ddfindex

Int64Index([0, 9, 18, 27], dtype='int64')

In [110]:
i = 0

In [111]:
dfd = ddf.loc[(ddfindex[i]+1):(ddfindex[i+1]-1)].reset_index(drop=True)
dfd.columns = ["项目名","营业收入(元)","收入比例(%)","营业利润(元)","利润比例(%)","毛利率(%)"]
dfd['日期'] = datetimes[i]

In [119]:
dfd

,项目名,营业收入(元),收入比例(%),营业利润(元),利润比例(%),毛利率(%),日期
0,工业制造业(行业),239.18亿,100.00,39.67亿,100.00,16.59,2024-06-30
1,消费类电池(产品),132.01亿,55.19,23.85亿,60.12,18.07,2024-06-30
2,电动汽车类电池(产品),62.01亿,25.93,7.23亿,18.23,11.66,2024-06-30
3,其他(产品),39.22亿,16.40,6.92亿,17.43,17.63,2024-06-30
4,储能系统类(产品),5.95亿,2.49,1.67亿,4.22,28.12,2024-06-30
5,国内(地区),142.33亿,59.51,30.73亿,77.47,21.59,2024-06-30
6,国外(地区),96.85亿,40.49,8.94亿,22.53,9.23,2024-06-30
7,直销(销售模式),239.18亿,100.00,39.67亿,100.00,16.59,2024-06-30


In [144]:
dfd[dfd['项目名'].str.contains('产品', na=False)]['毛利率(%)'].astype(float).sum()

75.48